# Fingerprint Enhancement System

**Mode A: Comparative & Enhancement Study**

Goal: improve degraded fingerprint quality while preserving ridge flow and supporting later ridge isolation and feature analysis. The experimental phase is complete; this notebook presents the frozen methodology and loads saved evidence.


# 1. Dataset

SOCOFing is organized as **Real**, **Altered Easy**, **Altered Medium**, and **Altered Hard**. Altered images are evaluated directly for structural changes; this project does not claim that every Altered image has a verified pixel-aligned clean Real reference.


# 2. Neutral Preprocessing P0

P0 performs grayscale conversion, consistent resizing when required, and 1st/99th-percentile intensity normalization. It contains no denoising, CLAHE, TV, Gabor filtering, or morphology.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from Fingerprint_Enhancement_System import *
ROOT = Path.cwd()
OUT = ROOT / 'outputs'
print('Final core loaded; large validations will be read from saved evidence.')


# 3. Conventional Baselines

Gaussian, Median, CLAHE, Contrast Stretching, Ordinary Gabor, and Wiener are conventional baselines/comparators. CLAHE remains useful, but it is **not** Member 1's final technique.


# 4. Member 1 — Non-Local Means Denoising

NLM compares image patches within a search neighbourhood and forms similarity-weighted estimates. The ridge-preserving final implementation is OpenCV \`fastNlMeansDenoising\`, with frozen normalized **h = 0.06**, uint8 rounding, template window 7, and search window 21.


# 5. Member 2 — Modified / Orientation-Adaptive Gabor

This fingerprint-specific method estimates local ridge orientation and frequency, selects responses from an adaptive Gabor bank, and confidence-blends them with the input. It is distinct from the fixed-frequency Ordinary Gabor baseline.


# 6. Member 3 — Total Variation Restoration

TV restoration uses regularization to suppress noise while discouraging unnecessary intensity variation across important ridge/edge boundaries. The controlled comparison used frozen weight 0.02.


# 7. Member 4 — Coherence-Guided Directional Diffusion

This method estimates ridge orientation and local coherence, then preferentially smooths along coherent ridge flow instead of equally in every direction. Frozen settings: four iterations, step 0.16.


# 8. Controlled Member Comparison

Saved 12-image controlled evidence; no experiment is rerun.


In [ ]:
member = pd.read_csv(OUT / 'member4_controlled_summary.csv')
member = member[member['Is member technique'].astype(str).str.lower().eq('true')]
display(member[['Member','Technique','MSE_mean','PSNR_mean','SSIM_mean']].sort_values('PSNR_mean', ascending=False).style.format({'MSE_mean':'{:.8f}','PSNR_mean':'{:.4f}','SSIM_mean':'{:.4f}'}))


# 9. Hybrid / Compatibility Investigation

Saved evidence shows: NLM 29.4366 dB; TV → NLM 28.9436 dB; NLM → Directional Diffusion 28.2968 dB; NLM → Modified Gabor 25.2277 dB; TV → NLM → Directional Diffusion 26.6099 dB. **No tested hybrid outperformed NLM alone**, so the simpler NLM pipeline was retained.


In [ ]:
compat = pd.read_csv(OUT / 'member_compatibility_summary.csv')
display(compat[compat['Category'].eq('Controlled')][['Pipeline','MSE_mean','PSNR_mean','SSIM_mean']].style.format({'MSE_mean':'{:.8f}','PSNR_mean':'{:.4f}','SSIM_mean':'{:.4f}'}))


# 10. Fresh Hold-Out Validation

The separate fresh hold-out used 100 Real images. Mean PSNR was **29.6443 dB**, and **99/100** images exceeded 28.17 dB. This is the strongest fresh generalisation evidence.


In [ ]:
holdout = json.loads((OUT / 'final_holdout_statistics.json').read_text())
pd.DataFrame([{'Images':holdout['enhanced_psnr']['count'],'Mean PSNR (dB)':holdout['enhanced_psnr']['mean'],'Above 28.17 dB':holdout['number_above_28_17']}])


# 11. Final 500-Image Robustness Validation

This was a **large-dataset robustness evaluation**, not 500 fresh unseen test images. Mean PSNR was **29.6303 dB**, a **+1.4603 dB** margin over 28.17 dB; **500/500** exceeded the benchmark. Difference from the fresh hold-out: **−0.0140 dB**.


In [ ]:
large = json.loads((OUT / 'final_large_real_statistics.json').read_text())
pd.DataFrame([{'Images':large['enhanced_psnr']['count'],'Mean PSNR (dB)':large['enhanced_psnr']['mean'],'Mean MSE':large['enhanced_mse']['mean'],'Mean SSIM':large['enhanced_ssim']['mean'],'Above benchmark':large['number_above_28_17']}])


# 12. Original Altered Validation

Across 1,500 original Altered fingerprints, ridge coherence and fragmentation improved overall, with a mild contrast reduction. Severity-specific saved values are loaded below; no pixel-aligned clean-reference claim is made.


In [ ]:
altered = pd.read_csv(OUT / 'final_large_altered_summary.csv')
display(altered[['severity','count','mean_delta_coherence','mean_delta_fragmentation','mean_delta_contrast']].style.format({'mean_delta_coherence':'{:+.4f}','mean_delta_fragmentation':'{:+.4f}','mean_delta_contrast':'{:+.4f}'}))


# 13. Final Recommended Enhancement Pipeline

**Fingerprint Input → P0 Neutral Preprocessing → OpenCV Non-Local Means (h = 0.06) → Enhanced Grayscale Fingerprint**

A small demonstration may use \`apply_final_enhancement\`; it never reruns validation.


In [ ]:
samples = sorted((ROOT/'data'/'SOCOFing'/'Real').glob('*.BMP'))[:1]
if samples:
    original = load_fingerprint(samples[0]); enhanced, metadata = apply_final_enhancement(original, assume_preprocessed=True)
    fig, axes = plt.subplots(1,2,figsize=(8,4))
    for ax,img,title in zip(axes,[original,enhanced],['P0 input','Final NLM h=0.06']): ax.imshow(img,cmap='gray',vmin=0,vmax=1); ax.set_title(title); ax.axis('off')
    plt.show(); display(metadata)
else: print('Dataset not found; saved validation tables above remain available.')


# 14. Downstream Structural Analysis

Enhanced grayscale output is passed separately to **segmentation → morphological processing → thinning/medial axis → ridge/minutiae analysis**. A skeleton is a downstream representation, not the final grayscale enhancement output.


# 15. Conclusion

NLM was strongest among the four principal member techniques; TV was second. Tested hybrids did not improve on NLM. Fresh hold-out evidence supports generalisation, the 500-image run supports robustness, and original Altered evaluation found structural benefits with a mild contrast trade-off. These results support the selected protocol, not universal superiority.
